# Chapter 4: The Coffee Lab — Finding the Sweet Spot

This notebook accompanies **Chapter 4** of the lecture notes.

**Agenda**

🔢 · 🔍 · 🚀 · ✂️ · 📈 · 🏁

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import check_brute_root, check_newton, check_secant

## 🔢 The Problem: Finding the Target Extraction Time

In your coffee lab, the extraction yield depends on brewing time. After subtracting a target yield, you get a function f(x) that is positive when you over-extract and negative when you under-extract. The extraction times where f(x) = 0 are the sweet spots — the times that hit your target yield exactly.

> Most real extraction curves have no closed-form solution. If you cannot solve f(x) = 0 by hand, how do you decide where to start looking — and how do you know when to stop?

<details><summary>Thought</summary>

You need two things: an initial region or guess that is "close enough" to a target crossing time, and a stopping rule that balances precision against computational cost. A brute-force scan gives you the region; iterative methods like Newton or secant then zoom in with far fewer evaluations. You stop when the change between iterates drops below a chosen tolerance.
</details>

We model the extraction curve (after subtracting the target) as f(x) = x³ − x, where x represents a normalised brewing time. This simplified curve crosses zero at three times: x = −1, x = 0, and x = 1. Let's visualize it.

In [ ]:
def f(x):
    """Target function: x^3 - x."""
    return x**3 - x


def f_prime(x):
    """Derivative of f: 3x^2 - 1."""
    return 3 * x**2 - 1


# Plot f(x) with roots marked
xs = np.linspace(-1.5, 1.5, 300)
ys = f(xs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, ys, color=_GOLDEN, ls='-', linewidth=1.2)
ax.axhline(0, color=_BORDER, linewidth=0.5, linestyle='--')

roots = np.array([-1.0, 0.0, 1.0])
ax.scatter(roots, f(roots), c=_TERRA, s=40, zorder=5, linewidths=0)
for r in roots:
    ax.annotate(f'x = {r:.0f}', (r, 0), textcoords='offset points',
                xytext=(8, 12), fontsize=9)

ax.set_xlabel('x')
ax.set_ylabel('f(x)')
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- The extraction curve crosses zero at three times: x = −1, 0, and 1. Each is a brewing time that hits the target yield.
- Between the roots, f(x) has local extrema where f'(x) = 0. These will matter for Newton's method — they are times where the extraction rate momentarily stalls.
- We will target the root at **x = 1** for our exercises, starting from the interval [0.5, 2.0].

### 🔍 Brute-Force Root Finding

> A grid search evaluates the extraction curve at n equally spaced time points and picks the one closest to zero. If you double n, the grid spacing halves and your best approximation can at most improve by half a grid step. Is there a fundamental limit to the accuracy you can achieve this way, and how does the cost scale with the desired precision?

<details><summary>Thought</summary>

The error is bounded by the grid spacing h = (b − a) / n. To cut the error in half you must double n, so the cost grows linearly with 1/h. Achieving 10 decimal places of accuracy on the target crossing time would require roughly 10 billion grid points — brute force does not scale to high precision.
</details>

Implement a function that scans n equally spaced time points in [a, b] and returns the time where |f(x)| is smallest — i.e., the time closest to hitting the target yield. Revisit the lecture notes for context on grid search.

Useful operations: `np.linspace()`, `np.abs()`, `np.argmin()`.

In [ ]:
def brute_force_root(f, a, b, n):
    """Return the x in [a, b] where |f(x)| is smallest over n grid points."""
    # YOUR CODE HERE
    pass


check_brute_root(brute_force_root, f, 0.5, 2.0, 100)

### 🚀 Newton's Method

> Newton's method uses the tangent line at the current guess to leap toward the target crossing time. Each step requires both f(x) (how far off the extraction yield is) and f'(x) (how fast the yield is changing). The convergence is quadratic — the number of correct digits roughly doubles each iteration. But what happens when f'(x) is close to zero at your current guess?

<details><summary>Thought</summary>

When f'(x) is near zero, the extraction rate is momentarily flat — the tangent line is nearly horizontal and the next iterate shoots far away. For our extraction curve, f'(x) = 3x² − 1 = 0 at x = ±1/√3 ≈ ±0.577. Starting near those brewing times can cause the method to diverge or oscillate wildly before (possibly) settling down.
</details>

Implement Newton's method. It should return a tuple: the final root estimate and a list of all iterates (the history). Revisit the lecture notes for the update rule.

Useful operations: `abs()` for the convergence check, list `.append()` for building the history.

In [ ]:
def newton_method(f, f_prime, x0, tol=1e-8, max_iter=100):
    """Return (root, history) using Newton's method."""
    # YOUR CODE HERE
    pass


check_newton(newton_method, f, f_prime, 1.4, true_root=1.0)

### ✂️ Secant Method

> The secant method replaces the exact derivative with a finite-difference approximation using the two most recent time estimates. You lose the quadratic convergence of Newton — the order drops to roughly 1.618 (the golden ratio). In exchange, you never need to know the rate of change of the extraction curve analytically. When would this trade-off be worth it?

<details><summary>Thought</summary>

When the derivative of the extraction curve is expensive or impossible to compute analytically — for instance, when f(x) comes from a physical brewing simulation or a black-box lab measurement. Each secant step costs one function evaluation (vs. two for Newton: f and f'), so even though it converges more slowly per step, the cost per function evaluation can be lower overall.
</details>

Implement the secant method. It needs two starting time estimates x0 and x1. Return a tuple: the final root estimate and the history of all iterates (starting with x0 and x1). Revisit the lecture notes for the update formula.

Useful operations: `abs()` for the convergence check, list `.append()` for building the history.

In [ ]:
def secant_method(f, x0, x1, tol=1e-8, max_iter=100):
    """Return (root, history) using the secant method."""
    # YOUR CODE HERE
    pass


check_secant(secant_method, f, 1.4, 1.2, true_root=1.0)

### 📈 Convergence Analysis

> We claimed Newton converges quadratically and the secant method at order ~1.618. But what does that actually look like when hunting for the optimal brewing time? If you plot the error on a log scale, a method of order p produces a curve whose slope steepens with each step — linear convergence gives a straight line, while quadratic convergence bends sharply downward.

<details><summary>Thought</summary>

On a log-scale error plot, each Newton step roughly doubles the number of correct decimal places in the brewing time, producing a curve that plunges nearly vertically after a few iterations. The secant method also accelerates, but more gently. The brute-force result is a single horizontal point — there is no iteration to improve it.
</details>

Run the cell below to compare all three methods visually. The true target crossing time is x = 1.

In [ ]:
true_root = 1.0

# Gather results
result_newton = newton_method(f, f_prime, 1.4)
result_secant = secant_method(f, 1.4, 1.2)
x_brute = brute_force_root(f, 0.5, 2.0, 100)

fig, ax = plt.subplots(figsize=(8, 5))

if result_newton is not None:
    _, history_newton = result_newton
    errors_newton = [abs(x - true_root) for x in history_newton]
    # Replace exact zeros with a tiny value for log scale
    errors_newton = [e if e > 0 else 1e-16 for e in errors_newton]
    ax.plot(range(len(errors_newton)), errors_newton,
            'o-', color=_ACCENT, markersize=5, linewidth=1.2,
            label='Newton')

if result_secant is not None:
    _, history_secant = result_secant
    errors_secant = [abs(x - true_root) for x in history_secant]
    errors_secant = [e if e > 0 else 1e-16 for e in errors_secant]
    ax.plot(range(len(errors_secant)), errors_secant,
            's-', color=_ORANGE, markersize=5, linewidth=1.2,
            label='Secant')

if x_brute is not None:
    brute_error = abs(x_brute - true_root)
    if brute_error == 0:
        brute_error = 1e-16
    ax.axhline(brute_error, color=_TERRA, linewidth=1, linestyle='--',
               label=f'Brute force (n=100)')

ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('|x_k - root|')
ax.legend(frameon=False, fontsize=9, labelcolor=_TEXT)
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- Newton's method (blue) converges in very few iterations. The error in the brewing time plunges nearly vertically on the log scale — that is quadratic convergence in action.
- The secant method (orange) also converges quickly, but the curve bends less aggressively.
- The brute-force baseline (red dashed line) is limited by the time-grid spacing. No amount of patience improves it beyond that resolution without increasing n.

### 🏁 Recap

**What we did:**
- 🔢 Visualized the coffee extraction curve f(x) = x³ − x and identified the three brewing times that hit the target yield.
- 🔍 Implemented brute-force root finding by scanning a time grid.
- 🚀 Implemented Newton's method using the exact derivative of the extraction curve.
- ✂️ Implemented the secant method using finite-difference approximations of the extraction rate.
- 📈 Compared convergence rates on a log-scale error plot.

**Key takeaways:**
- Brute force is simple but its accuracy is capped by the time-grid resolution, and the cost scales linearly with desired precision.
- Newton's method converges quadratically but requires the derivative and a good starting guess for the brewing time.
- The secant method trades the derivative for one extra starting point, converging at order ~1.618 — a practical choice when the extraction rate is not available analytically.

**Now head back for self-check questions and key learnings in the lecture notes.**